In [1]:
# 12_outcome_proxy_v1c_beta_sensitivity.ipynb

from __future__ import annotations

from pathlib import Path
import sys
import numpy as np
import pandas as pd

# --- resolve repo root (notebooks/ -> repo root) ---
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name.lower() == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

assert (REPO_ROOT / "src").exists(), f"Expected src/ at repo root: {REPO_ROOT}"

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repo root:", REPO_ROOT)

from src.rtm.io_hazard import load_rtm_pluvial_v1_buildings

OUT_DIR = REPO_ROOT / "outputs" / "rtm" / "outcomes"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Outcome outputs:", OUT_DIR)

Repo root: C:\Users\C.Price\Habnetic\resilient-housing-bayes
Outcome outputs: C:\Users\C.Price\Habnetic\resilient-housing-bayes\outputs\rtm\outcomes


In [2]:
# --- Load exposure ---
E_PATH = REPO_ROOT / "outputs" / "rtm" / "water_exposure_Ehat_v0.parquet"
if not E_PATH.exists():
    raise FileNotFoundError(f"Missing E_hat parquet at {E_PATH}")

E = pd.read_parquet(E_PATH)

if "bldg_id" not in E.columns:
    raise ValueError(f"E_hat table missing bldg_id. Columns: {list(E.columns)}")

if "E_hat" not in E.columns:
    candidates = [c for c in E.columns if c.lower() in {"ehat", "e_hat", "water_exposure_ehat", "exposure"}]
    if candidates:
        E = E.rename(columns={candidates[0]: "E_hat"})
    else:
        raise ValueError(f"Could not find E_hat column. Columns: {list(E.columns)}")

E = E[["bldg_id", "E_hat"]].copy()
print("E rows:", len(E), "unique bldg_id:", E["bldg_id"].is_unique)
E.head()

E rows: 221324 unique bldg_id: True


,bldg_id,E_hat
0,305012,0.536838
1,313960,0.677579
2,313263,0.251841
3,310491,0.189019
4,313127,-0.292821


In [3]:
# --- Load hazard (Phase 1a frozen) ---
haz = load_rtm_pluvial_v1_buildings()
print("Haz rows:", len(haz), "unique bldg_id:", haz["bldg_id"].is_unique)
haz.head()

Haz rows: 221324 unique bldg_id: True


,bldg_id,H_pluvial_v1_mm
0,305012,25.422161
1,313960,25.418823
2,313263,25.423113
3,310491,25.424500
4,313127,25.423491


In [4]:
# --- Merge (full city) ---
df = E.merge(haz, on="bldg_id", how="inner", validate="one_to_one")
print("Joined rows:", len(df))
print("Drops vs E:", len(E) - len(df))
print("Drops vs hazard:", len(haz) - len(df))

# Hard sanity (this is your RTM row count)
assert len(df) == 221_324, "Unexpected join row count; investigate before generating outcome."
assert df["E_hat"].notna().all()
assert df["H_pluvial_v1_mm"].notna().all()

df.head()

Joined rows: 221324
Drops vs E: 0
Drops vs hazard: 0


,bldg_id,E_hat,H_pluvial_v1_mm
0,305012,0.536838,25.422161
1,313960,0.677579,25.418823
2,313263,0.251841,25.423113
3,310491,0.189019,25.424500
4,313127,-0.292821,25.423491


In [5]:
def zscore(x: pd.Series) -> pd.Series:
    x = x.astype(float)
    mu = x.mean()
    sd = x.std(ddof=1)
    if sd <= 0:
        raise ValueError("Standard deviation is zero; cannot z-score.")
    return (x - mu) / sd

df["E_std"] = zscore(df["E_hat"])
df["H_std"] = zscore(df["H_pluvial_v1_mm"])

print("E_std mean/std:", float(df["E_std"].mean()), float(df["E_std"].std(ddof=1)))
print("H_std mean/std:", float(df["H_std"].mean()), float(df["H_std"].std(ddof=1)))

E_std mean/std: 4.4945863533287865e-19 0.9999999999999999
H_std mean/std: 1.561788497311158e-15 0.9999999999999999


In [6]:
# --- v1c settings: fixed baseline rate, vary betas ---
seed = 42
rng = np.random.default_rng(seed)

p0_baseline = 0.05
alpha = float(np.log(p0_baseline / (1.0 - p0_baseline)))  # baseline logit

# Three beta scenarios (edit if you want, but keep names stable)
SCENARIOS = [
    {"scenario": "bE10_bH06", "beta_E": 1.0, "beta_H": 0.6},
    {"scenario": "bE10_bH02", "beta_E": 1.0, "beta_H": 0.2},
    {"scenario": "bE02_bH06", "beta_E": 0.2, "beta_H": 0.6},
]

print("alpha:", alpha)
print("scenarios:", SCENARIOS)

alpha: -2.9444389791664403
scenarios: [{'scenario': 'bE10_bH06', 'beta_E': 1.0, 'beta_H': 0.6}, {'scenario': 'bE10_bH02', 'beta_E': 1.0, 'beta_H': 0.2}, {'scenario': 'bE02_bH06', 'beta_E': 0.2, 'beta_H': 0.6}]


In [7]:
def logistic(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))

def generate_and_save(df: pd.DataFrame, scenario: str, beta_E: float, beta_H: float) -> Path:
    linpred = alpha + beta_E * df["E_std"].values + beta_H * df["H_std"].values
    p_true = logistic(linpred).astype(np.float32)

    # Binary outcome
    Y = rng.binomial(n=1, p=p_true, size=len(df)).astype(np.int8)

    out = df[["bldg_id"]].copy()
    out["Y_damage"] = Y
    out["p_true"] = p_true
    out["outcome_src"] = "synthetic"
    out["outcome_version"] = f"v1c_{scenario}"
    out["seed"] = seed
    out["p0_baseline"] = float(p0_baseline)
    out["alpha"] = float(alpha)
    out["beta_E"] = float(beta_E)
    out["beta_H"] = float(beta_H)

    out_path = OUT_DIR / f"Y_damage_v1c_{scenario}.parquet"
    out.to_parquet(out_path, index=False)

    # quick diagnostics
    realized_rate = float(out["Y_damage"].mean())
    print(f"\nSaved: {out_path}")
    print(f"Scenario {scenario}: realized_rate={realized_rate:.4f}")
    print("p_true min/median/p95/max:",
          float(np.min(p_true)),
          float(np.median(p_true)),
          float(np.quantile(p_true, 0.95)),
          float(np.max(p_true)))

    # QA
    assert out["bldg_id"].is_unique
    assert set(out["Y_damage"].unique()).issubset({0, 1})
    assert out["Y_damage"].isna().sum() == 0
    assert out["p_true"].isna().sum() == 0

    return out_path

paths = []
for sc in SCENARIOS:
    paths.append(generate_and_save(df, sc["scenario"], sc["beta_E"], sc["beta_H"]))

print("\nAll saved:")
for p in paths:
    print(" -", p)


Saved: C:\Users\C.Price\Habnetic\resilient-housing-bayes\outputs\rtm\outcomes\Y_damage_v1c_bE10_bH06.parquet
Scenario bE10_bH06: realized_rate=0.0791
p_true min/median/p95/max: 0.0008382414816878736 0.057835884392261505 0.21316818222403522 0.7388597130775452

Saved: C:\Users\C.Price\Habnetic\resilient-housing-bayes\outputs\rtm\outcomes\Y_damage_v1c_bE10_bH02.parquet
Scenario bE10_bH02: realized_rate=0.0685
p_true min/median/p95/max: 0.0013341184239834547 0.06859880685806274 0.14941841214895246 0.44238242506980896

Saved: C:\Users\C.Price\Habnetic\resilient-housing-bayes\outputs\rtm\outcomes\Y_damage_v1c_bE02_bH06.parquet
Scenario bE02_bH06: realized_rate=0.0610
p_true min/median/p95/max: 0.012574066407978535 0.04506992921233177 0.14714281857013697 0.42859163880348206

All saved:
 - C:\Users\C.Price\Habnetic\resilient-housing-bayes\outputs\rtm\outcomes\Y_damage_v1c_bE10_bH06.parquet
 - C:\Users\C.Price\Habnetic\resilient-housing-bayes\outputs\rtm\outcomes\Y_damage_v1c_bE10_bH02.parquet